<a href="https://colab.research.google.com/github/isaacadebayo/Predictive-Analytics-Public-Datasets/blob/main/WHISPER_Audio_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration


# Load Parquet dataset using Hugging Face

In [32]:
from random import sample
dataset = load_dataset("PolyAI/minds14", name="en-US", split="train")


# Take the very first sample audio

In [33]:
sample_audio = dataset[0]["audio"]["array"]
sample_audio

array([ 0.        ,  0.00024414, -0.00024414, ..., -0.00024414,
        0.        ,  0.        ], dtype=float32)

In [44]:
import pandas as pd
df_parquet = pd.read_parquet('/content/minds14.parquet')
display(df_parquet.head())

,path,audio,transcription,english_transcription,intent_class,lang_id
0,en-US~JOINT_ACCOUNT/602ba55abb1e6d0fbce92065.wav,{'bytes': b'RIFF\xdeR\x01\x00WAVEfmt \x12\x00\...,I would like to set up a joint account with my...,I would like to set up a joint account with my...,11,4
1,en-US~JOINT_ACCOUNT/602baf24bb1e6d0fbce922a7.wav,{'bytes': b'RIFF2\xd0\x00\x00WAVEfmt \x12\x00\...,Henry County set up a joint account with my wi...,Henry County set up a joint account with my wi...,11,4
2,en-US~JOINT_ACCOUNT/602b9f97963e11ccd901cc32.wav,{'bytes': b'RIFF\x88\xf5\x02\x00WAVEfmt \x12\x...,hi I'd like to set up a joint account with my ...,hi I'd like to set up a joint account with my ...,11,4
3,en-US~JOINT_ACCOUNT/602bacab5f67b421554f6488.wav,{'bytes': b'RIFF\x88e\x00\x00WAVEfmt \x12\x00\...,how do I start a joint account,how do I start a joint account,11,4
4,en-US~JOINT_ACCOUNT/602b9d4cbb1e6d0fbce91fa4.wav,{'bytes': b'RIFF2x\x00\x00WAVEfmt \x12\x00\x00...,can you help me set up a joint bank account,can you help me set up a joint bank account,11,4


### 1. Prepare Data for Intent Classification

We need to extract the `transcription` as our features (X) and `intent_class` as our labels (y). We'll also encode the `intent_class` column into numerical labels, as machine learning models typically require numerical input for labels. Finally, we'll split the data into training and testing sets.

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Extract features (X) and labels (y)
X = df_parquet['transcription'].values
y = df_parquet['intent_class'].values

# Encode categorical labels into numerical format
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Display the mapping of original intent to numerical labels
print("Intent to numerical label mapping:")
for i, intent in enumerate(label_encoder.classes_):
    print(f"{intent}: {i}")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

Intent to numerical label mapping:
0: 0
1: 1
2: 2
3: 3
4: 4
5: 5
6: 6
7: 7
8: 8
9: 9
10: 10
11: 11
12: 12
13: 13

Training set size: 450
Testing set size: 113


### 2. Text Vectorization

Machine learning models cannot directly process raw text. We need to convert the text into numerical representations. TF-IDF (Term Frequency-Inverse Document Frequency) is a common technique that reflects the importance of a word in a document relative to a collection of documents.

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to 5000 for simplicity

# Fit the vectorizer on the training data and transform both training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

Shape of X_train_tfidf: (450, 602)
Shape of X_test_tfidf: (113, 602)


### 3. Train a Classification Model

We will use a Logistic Regression model, a simple yet effective algorithm for classification tasks. It will learn to map the TF-IDF features to the intent classes.

In [47]:
from sklearn.linear_model import LogisticRegression

# Initialize and train a Logistic Regression model
classifier = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence
classifier.fit(X_train_tfidf, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


### 4. Evaluate the Model

Now, let's evaluate the trained model's performance on the test set using various classification metrics.

In [50]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

# Make predictions on the test set
y_pred = classifier.predict(X_test_tfidf)

# Get the original string names for the classes from the dataset features
# Assuming 'dataset' object from earlier cells is available and contains 'intent_class' features
# If dataset is not available, one would manually create a list of class names.
class_names = datasets['train'].features['intent_class'].names

# Evaluate the model
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=class_names))

print("\n--- Overall Metrics ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Macro Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro Recall: {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro F1-Score: {f1_score(y_test, y_pred, average='macro'):.4f}")


--- Classification Report ---
                     precision    recall  f1-score   support

             abroad       0.80      1.00      0.89         4
            address       1.00      0.75      0.86         4
          app_error       0.88      0.88      0.88         8
          atm_limit       0.89      0.89      0.89         9
            balance       1.00      0.85      0.92        13
      business_loan       1.00      1.00      1.00        10
        card_issues       0.82      1.00      0.90         9
       cash_deposit       0.91      1.00      0.95        10
       direct_debit       1.00      1.00      1.00         5
             freeze       0.89      1.00      0.94         8
 high_value_payment       0.86      1.00      0.92         6
      joint_account       0.88      0.88      0.88         8
latest_transactions       1.00      0.78      0.88         9
           pay_bill       1.00      0.90      0.95        10

           accuracy                           0.92  

### 5. Evaluate Logistic Regression with `torchmetrics`

We will now re-evaluate the performance of our `LogisticRegression` model using `torchmetrics`. This involves converting the numpy arrays of predictions and true labels from `scikit-learn` into PyTorch tensors and then applying the `torchmetrics` classification functions.

In [ ]:
import torch
import torchmetrics

# Ensure device is set (from previous torchmetrics setup in cell UxVUifeS1ySf)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert numpy arrays from sklearn model to torch tensors and move to device
# y_test and y_pred are available from the LogisticRegression evaluation in cell 660f9a6c
true_labels_tensor = torch.tensor(y_test, dtype=torch.long).to(device)
predicted_labels_tensor = torch.tensor(y_pred, dtype=torch.long).to(device)

# NUM_CLASSES is defined in cell UxVUifeS1ySf

# Re-initialize torchmetrics for a clean evaluation of the LR model
# We use 'macro' average to ensure rarer intents are weighted fairly against common ones.
actual_accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(device)
actual_f1_metric = torchmetrics.F1Score(task="multiclass", num_classes=NUM_CLASSES, average="macro").to(device)
actual_precision_metric = torchmetrics.Precision(task="multiclass", num_classes=NUM_CLASSES, average="macro").to(device)
actual_recall_metric = torchmetrics.Recall(task="multiclass", num_classes=NUM_CLASSES, average="macro").to(device)

# Update the metrics with the actual predictions and true labels
actual_accuracy_metric.update(predicted_labels_tensor, true_labels_tensor)
actual_f1_metric.update(predicted_labels_tensor, true_labels_tensor)
actual_precision_metric.update(predicted_labels_tensor, true_labels_tensor)
actual_recall_metric.update(predicted_labels_tensor, true_labels_tensor)

# Compute and print the final metrics
print("\n--- Actual Intent Classification Performance Metrics (using torchmetrics) ---")
print(f"Accuracy:  {actual_accuracy_metric.compute().item() * 100:.2f}%")
print(f"Macro F1:  {actual_f1_metric.compute().item() * 100:.2f}%")
print(f"Precision: {actual_precision_metric.compute().item() * 100:.2f}%")
print(f"Recall:    {actual_recall_metric.compute().item() * 100:.2f}%")

# Reset the metrics after computation
actual_accuracy_metric.reset()
actual_f1_metric.reset()
actual_precision_metric.reset()
actual_recall_metric.reset()

# Downloaded Parquet on local drive

In [34]:
datasets = load_dataset("parquet", data_files={"train": "minds14.parquet"})

In [35]:
datasets

DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
        num_rows: 563
    })
})

In [36]:
sample_parquet = datasets["train"][0]["audio"]["array"]
sample_parquet

array([ 0.        ,  0.00024414, -0.00024414, ..., -0.00024414,
        0.        ,  0.        ], dtype=float32)

# Setup PyTorch and Whisper

In [37]:
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to(device)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [38]:
input_features = processor(sample_parquet, sampling_rate=16_000, return_tensors="pt").input_features
input_features = input_features.to(device)

# Generate text token

In [39]:
with torch.no_grad():
  predicted_ids = model.generate(input_features)

In [40]:
transcription = processor.batch_decode(
    predicted_ids, skip_special_tokens=True
                                        )
print(transcription[0])


 I would like to set up a joint account with my partner. How do I proceed with doing that?


### 6. Calculate Word Error Rate (WER) for Whisper Transcription

Now, let's calculate the Word Error Rate (WER) between the transcription generated by the Whisper model and the ground truth transcription from the `minds14.parquet` dataset. This metric will tell us how accurate the Whisper model's audio-to-text conversion is for the given sample.

In [51]:
# Install the 'evaluate' library if you haven't already
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.7 MB/s eta 0:00:00


In [52]:
import evaluate

# Load the WER metric
wer_metric = evaluate.load("wer")

# Get the ground truth transcription for the sample_parquet
# The sample_parquet was derived from datasets["train"][0]["audio"]["array"]
ground_truth_transcription = datasets["train"][0]["transcription"]

# The Whisper transcription is stored in the `transcription` variable from cell lkNTIBAezWvH
whisper_output = transcription[0]

# Calculate WER
# The `compute` method expects lists of predictions and references
wer_score = wer_metric.compute(predictions=[whisper_output], references=[ground_truth_transcription])

print(f"\n--- Word Error Rate (WER) between Whisper Transcription and Ground Truth ---")
print(f"Whisper Transcription: '{whisper_output}'")
print(f"Ground Truth Transcription: '{ground_truth_transcription}'")
print(f"WER: {wer_score:.4f}")


--- Word Error Rate (WER) between Whisper Transcription and Ground Truth ---
Whisper Transcription: ' I would like to set up a joint account with my partner. How do I proceed with doing that?'
Ground Truth Transcription: 'I would like to set up a joint account with my partner'
WER: 0.6667


In [41]:
! pip install torchmetrics

#### Torchmetics pytorch sample code for evaluation epoch

In [55]:
import torch
import torchmetrics

# Set up your device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define dataset parameters for MInDS-14
NUM_CLASSES = 14  # MInDS-14 has 14 specific intent categories

# Convert numpy arrays from sklearn model to torch tensors and move to device
# y_test and y_pred are available from the LogisticRegression evaluation in cell 660f9a6c
true_labels_tensor = torch.tensor(y_test, dtype=torch.long).to(device)
predicted_labels_tensor = torch.tensor(y_pred, dtype=torch.long).to(device)

# Initialize PyTorch-native Evaluation Metrics
# We use 'macro' average to ensure rarer intents are weighted fairly against common ones.
accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(device)
f1_metric = torchmetrics.F1Score(task="multiclass", num_classes=NUM_CLASSES, average="macro").to(device)
precision_metric = torchmetrics.Precision(task="multiclass", num_classes=NUM_CLASSES, average="macro").to(device)
recall_metric = torchmetrics.Recall(task="multiclass", num_classes=NUM_CLASSES, average="macro").to(device)

# Update the metrics with the actual predictions and true labels from Logistic Regression
accuracy_metric.update(predicted_labels_tensor, true_labels_tensor)
f1_metric.update(predicted_labels_tensor, true_labels_tensor)
precision_metric.update(predicted_labels_tensor, true_labels_tensor)
recall_metric.update(predicted_labels_tensor, true_labels_tensor)

In [56]:
# --- Calculate and Print Final Metric Aggregations ---
# Compute the final scores across all accumulated batches
final_accuracy = accuracy_metric.compute()
final_f1 = f1_metric.compute()
final_precision = precision_metric.compute()
final_recall = recall_metric.compute()

print("--- Actual Logistic Regression Intent Classification Performance Metrics (using torchmetrics) ---")
print(f"Accuracy:  {final_accuracy.item() * 100:.2f}%")
print(f"Macro F1:  {final_f1.item() * 100:.2f}%")
print(f"Precision: {final_precision.item() * 100:.2f}%")
print(f"Recall:    {final_recall.item() * 100:.2f}%")

# Reset the metrics after computation
accuracy_metric.reset()
f1_metric.reset()
precision_metric.reset()
recall_metric.reset()

--- Actual Logistic Regression Intent Classification Performance Metrics (using torchmetrics) ---
Accuracy:  92.04%
Macro F1:  91.72%
Precision: 92.23%
Recall:    92.23%


### 7. Compare Whisper Transcription Quality Across Different Intent Classes

Now, let's analyze how well Whisper transcribes audio for different intent classes. We will iterate through a subset of the dataset, get Whisper's transcription for each audio sample, and calculate the Word Error Rate (WER) against the ground truth transcription. This will help us identify if Whisper performs differently depending on the intent.

In [58]:
from tqdm.notebook import tqdm

# Reload WER metric (if not already loaded)
wer_metric = evaluate.load("wer")

# Initialize lists to store results
wer_results = []

# Create a DataFrame from the dataset for easier grouping
df_dataset = pd.DataFrame(datasets['train'])

# Get unique intent classes
unique_intents = df_dataset['intent_class'].unique()

# Dictionary to store WERs per intent
wer_by_intent = {intent: [] for intent in unique_intents}

# Limit the number of samples per intent for faster execution during demonstration
samples_per_intent_limit = 5 # Set to None or a very large number to process all samples

for intent_id in tqdm(unique_intents, desc="Processing Intents"):
    intent_name = label_encoder.inverse_transform([intent_id])[0]

    # Filter samples for the current intent
    intent_samples = df_dataset[df_dataset['intent_class'] == intent_id]

    if samples_per_intent_limit:
        intent_samples = intent_samples.head(samples_per_intent_limit)

    for index, row in intent_samples.iterrows():
        audio_array = row['audio']['array']
        ground_truth = row['transcription']

        # Process audio with Whisper
        input_features = processor(audio_array, sampling_rate=16_000, return_tensors="pt").input_features
        input_features = input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(input_features)
        whisper_transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

        # Calculate WER
        wer = wer_metric.compute(predictions=[whisper_transcription], references=[ground_truth])
        wer_by_intent[intent_id].append(wer)


print("\n--- Average WER per Intent Class ---")
for intent_id, wers in wer_by_intent.items():
    if wers:
        avg_wer = sum(wers) / len(wers)
        intent_name = label_encoder.inverse_transform([intent_id])[0]
        print(f"Intent '{intent_name}': Average WER = {avg_wer:.4f} (based on {len(wers)} samples)")
    else:
        intent_name = label_encoder.inverse_transform([intent_id])[0]
        print(f"Intent '{intent_name}': No samples processed.")

Processing Intents:   0%|          | 0/14 [00:00<?, ?it/s]


--- Average WER per Intent Class ---
Intent '11': Average WER = 0.8048 (based on 5 samples)
Intent '4': Average WER = 0.5636 (based on 5 samples)
Intent '13': Average WER = 0.6809 (based on 5 samples)
Intent '9': Average WER = 0.7920 (based on 5 samples)
Intent '2': Average WER = 0.5308 (based on 5 samples)
Intent '1': Average WER = 0.4329 (based on 5 samples)
Intent '3': Average WER = 0.4814 (based on 5 samples)
Intent '0': Average WER = 0.4208 (based on 5 samples)
Intent '8': Average WER = 0.6072 (based on 5 samples)
Intent '12': Average WER = 0.6726 (based on 5 samples)
Intent '10': Average WER = 0.4831 (based on 5 samples)
Intent '7': Average WER = 0.5606 (based on 5 samples)
Intent '6': Average WER = 0.6463 (based on 5 samples)
Intent '5': Average WER = 0.5457 (based on 5 samples)


### Word Error Rate (WER) with Whisper Transformer:


1.   Purpose: WER is specifically used to evaluate the performance of an Audio-to-Text (A2T) model, like Whisper.

2.   When to use: You use WER when you want to measure how accurately your Whisper model (or any speech-to-text system) converts spoken audio into written text. It compares the generated transcription to a human-annotated "ground truth" transcription. A lower WER indicates better transcription quality.




### Scikit-learn Classification Report:



1.   Purpose: This report is used to evaluate the performance of a classification model (like our LogisticRegression model) trained using scikit-learn.
2.   When to use: You use scikit-learn's classification_report when your model predicts categorical labels (e.g., intent classes) from input features (like TF-IDF vectorized text).



1.   It provides detailed metrics (Precision, Recall, F1-score, Support) for each class and overall averages, which are crucial for understanding how well your model distinguishes between different categories.
2.   In this notebook, it waas used to evaluate the intent classification model's ability to predict the correct intent based on the text transcription.





### Torchmetrics:



1.   Purpose: torchmetrics offers a comprehensive suite of metrics for evaluating PyTorch-based models, including classification metrics like Accuracy, F1-score, Precision, and Recall.

1.   When to use: While you can adapt it to evaluate scikit-learn model outputs (as we just did by converting y_pred and y_test to tensors), its primary strength lies in being integrated directly into PyTorch training and validation loops.
2.   It's designed for efficient, GPU-accelerated metric calculation in deep learning contexts, allowing you to easily track and accumulate metrics across batches and epochs.


2.   In this notebook: It was used in a simulated PyTorch evaluation, and then demonstrated how it could be applied to your scikit-learn model's outputs, showcasing its flexibility, especially if you were to transition to a PyTorch-based intent classification model in the future.




